# Sentiment Classifier Training — SemEval 2017 / tweet_eval

Fine-tunes **DistilBERT** (Sanh et al., 2019) on the **SemEval 2017 Tweet Sentiment** corpus  
(accessed via `cardiffnlp/tweet_eval` on HuggingFace, N=45,615 tweets).  

### Steps before running
1. `Runtime → Change runtime type → T4 GPU` → Save
2. Run **Cell 1 only** first (installs packages + restarts kernel automatically)
3. After restart, run **all remaining cells** from Cell 2 onward

In [ ]:
# ── CELL 1 — Install packages  (run this cell first, alone) ───────────────────
# Pins datasets to 3.x to avoid a torchvision VideoReader conflict in Colab 4.x
!pip install "datasets==3.2.0" "transformers>=4.40.0" accelerate scikit-learn -q
print('Packages installed.')
print('Restarting kernel automatically...')

# Restart the kernel so the pinned versions are active
import os
os.kill(os.getpid(), 9)

In [ ]:
# ── CELL 2 — Check GPU  (run after kernel restarts) ───────────────────────────
import torch
print('GPU available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device        :', torch.cuda.get_device_name(0))
    print('Memory (GB)   :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('WARNING: No GPU. Go to Runtime → Change runtime type → T4 GPU')

import datasets, transformers
print('datasets version    :', datasets.__version__)
print('transformers version:', transformers.__version__)

In [ ]:
# ── CELL 3 — Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_PATH = '/content/drive/MyDrive/sentiment_classifier'
os.makedirs(SAVE_PATH, exist_ok=True)
print('Model will be saved to:', SAVE_PATH)

In [ ]:
# ── CELL 4 — Load SemEval 2017 Tweet Sentiment dataset ────────────────────────
from datasets import load_dataset
from collections import Counter

dataset = load_dataset('cardiffnlp/tweet_eval', 'sentiment')

ID2LABEL = {0: 'negative', 1: 'neutral', 2: 'positive'}
LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}

print(f'Train : {len(dataset["train"]):,} samples')
print(f'Val   : {len(dataset["validation"]):,} samples')
print(f'Test  : {len(dataset["test"]):,} samples')
print()
counts = Counter(dataset['train']['label'])
print('Label distribution (train):')
for lid, name in ID2LABEL.items():
    pct = counts[lid] / len(dataset['train']) * 100
    print(f'  {name:10s}: {counts[lid]:,}  ({pct:.1f}%)')
print()
print('Sample:', dataset['train'][0])

In [ ]:
# ── CELL 5 — Tokenise ─────────────────────────────────────────────────────────
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=128,
        padding='max_length'
    )

tokenised = dataset.map(tokenize, batched=True, batch_size=512)
tokenised = tokenised.rename_column('label', 'labels')
tokenised.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print('Tokenisation complete.')
print('Input shape (first sample):', tokenised['train'][0]['input_ids'].shape)

In [ ]:
# ── CELL 6 — Load DistilBERT with 3-class classification head ─────────────────
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total:,}')
print(f'Trainable parameters: {trainable:,}')

In [ ]:
# ── CELL 7 — Train  (~8-12 min on T4 GPU) ────────────────────────────────────
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1_per_class = f1_score(labels, preds, average=None, labels=[0, 1, 2])
    return {
        'accuracy'    : round(accuracy_score(labels, preds), 4),
        'f1_macro'    : round(f1_score(labels, preds, average='macro'), 4),
        'f1_negative' : round(float(f1_per_class[0]), 4),
        'f1_neutral'  : round(float(f1_per_class[1]), 4),
        'f1_positive' : round(float(f1_per_class[2]), 4),
    }

# warmup_steps = 10% of total steps
# total_steps  = (45615 / 64) * 3 epochs ≈ 2138
WARMUP_STEPS = 200

training_args = TrainingArguments(
    output_dir='/content/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=50,
    report_to='none',
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenised['train'],
    eval_dataset=tokenised['validation'],
    compute_metrics=compute_metrics,
)

print('Starting training...')
trainer.train()

In [ ]:
# ── CELL 8 — Evaluate on held-out test set ────────────────────────────────────
test_results = trainer.evaluate(tokenised['test'])

print('=' * 55)
print('  TEST SET RESULTS  —  cite these in your thesis')
print('=' * 55)
print(f"  Accuracy    : {test_results.get('eval_accuracy', 0):.4f}")
print(f"  F1 Macro    : {test_results.get('eval_f1_macro', 0):.4f}")
print(f"  F1 Negative : {test_results.get('eval_f1_negative', 0):.4f}")
print(f"  F1 Neutral  : {test_results.get('eval_f1_neutral', 0):.4f}")
print(f"  F1 Positive : {test_results.get('eval_f1_positive', 0):.4f}")
print('=' * 55)

In [ ]:
# ── CELL 9 — Save model + tokeniser to Google Drive ───────────────────────────
import json

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

metrics = {
    'base_model'   : 'distilbert-base-uncased',
    'dataset'      : 'cardiffnlp/tweet_eval (sentiment, SemEval 2017)',
    'train_size'   : len(dataset['train']),
    'val_size'     : len(dataset['validation']),
    'test_size'    : len(dataset['test']),
    'epochs'       : 3,
    'learning_rate': 2e-5,
    'warmup_steps' : WARMUP_STEPS,
    'test_results' : {k.replace('eval_', ''): v for k, v in test_results.items()},
}
with open(f'{SAVE_PATH}/training_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Saved to Google Drive:', SAVE_PATH)
print('Files:')
for fname in sorted(os.listdir(SAVE_PATH)):
    size = os.path.getsize(f'{SAVE_PATH}/{fname}')
    print(f'  {fname:40s}  {size/1e6:.1f} MB')

In [ ]:
# ── CELL 10 — Download model as ZIP ──────────────────────────────────────────
import shutil
from google.colab import files

zip_path = '/content/sentiment_classifier'
shutil.make_archive(zip_path, 'zip', SAVE_PATH)

zip_size = os.path.getsize(f'{zip_path}.zip') / 1e6
print(f'ZIP size: {zip_size:.0f} MB')
print('Starting download...')
files.download(f'{zip_path}.zip')

## After downloading

1. Unzip `sentiment_classifier.zip`
2. Place the folder at:
   ```
   m3_implementation/memory/models/sentiment_classifier/
   ```
3. System auto-detects and uses your trained model on next startup — no other changes needed.

`training_metrics.json` inside the folder has the test results to cite in your thesis.